# Lab: Telco Customer Churn Prediction
**Author:** Gordan  
**Role:** Machine Learning Intern  

---

## 1. Scenario & Background
Customer churn occurs when subscribers stop doing business with a service provider. In the highly competitive telecommunications sector, acquiring new customers costs significantly more than retaining existing ones. 

The objective of this project is to build a predictive machine learning pipeline using a **K-Nearest Neighbors (KNN)** classifier. By identifying high-risk customers before they defect, the business can deploy targeted retention strategies to protect revenue.

### Learning Objectives
* **Data Auditing:** Identify and resolve structural defects (e.g., string-to-numeric anomalies, missing data values).
* **Feature Engineering:** Transform mixed categorical feature fields into numeric dimensions using One-Hot Encoding.
* **Feature Scaling:** Synchronize measurement scales to eliminate metric space distortions during distance calculation.
* **Optimization:** Evaluate a spectrum of hyperparameter configurations ($K$-values) to find the optimal trade-off between Precision and Recall.

---

## 2. Dataset Overview
* **Source:** Kaggle Telco Customer Churn Dataset
* **Instance Count:** 7,043 customers
* **Feature Scope:** 20 predictor columns (demographics, signed services, account metrics)
* **Target Variable:** `Churn` (Yes / No)

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Load the Telco Churn dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("=== Dataset Shape ===")
print(df.shape)

print("\n=== Data Types Overview ===")
print(df.dtypes)

print("\n=== Target Class Distribution ===")
print(df['Churn'].value_counts())

# 1. Drop customerID since it's just an arbitrary tracker
df.drop('customerID', axis=1, inplace=True)

# 2. Convert TotalCharges to numeric. 'coerce' turns hidden blank spaces into NaN (missing values)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 3. Check how many missing values were created from those blank spaces
print(f"Missing values found in TotalCharges: {df['TotalCharges'].isnull().sum()}")

# 4. Fill those missing values with the median of the column so the model doesn't crash
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print("\n=== Cleaned Numerical Columns ===")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].dtypes)

# Step 2: Data Preprocessing - Encoding Categorical Variables

# 1. Map target variable 'Churn' to binary numeric values: Yes = 1, No = 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Separate features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# 3. Convert all categorical text columns into numeric indicators
# drop_first=True prevents multicollinearity (the dummy variable trap)
X = pd.get_dummies(X, drop_first=True)

# 4. Ensure all resulting columns are numeric integers/floats instead of booleans
X = X.astype(float)

print("=== Encoding Complete ===")
print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")



=== Dataset Shape ===
(7043, 21)

=== Data Types Overview ===
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

=== Target Class Distribution ===
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Missing values found in TotalCharges: 11

=== Cleaned Numerical Columns ===
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object
=== Encoding Complete ===
Features shape (X): (7043, 30)
Target shape (y): (7